# List file annotation edges

Annotations written by **cdf_file_annotation** are `CogniteDiagramAnnotation` edges (`cdf_cdm` / `v1`). The detected text, status, confidence, page, and bounding box are properties on the edge.

- **Direct matches** (`diagrams.AssetLink`, `diagrams.FileLink`) start at the file and end at the matched asset or file. The edge is stored in the file instance space.
- **Pattern detections** start at the file and end at the pattern sink node until promote links them to an entity. Those edges are stored in the pattern-mode instance space. Rejected pattern edges are removed from the data model after promote; the RAW pattern table keeps the audit row.

Credentials come from a `.env` file: `CDF_PROJECT`, `CDF_CLUSTER`, `IDP_CLIENT_ID`, `IDP_CLIENT_SECRET`, `IDP_TENANT_ID`. The space and view defaults match `config.dev.yaml` for this module.

In [120]:
import json
import os
from collections.abc import Sequence
from pprint import pprint

import pandas as pd
from cognite.client import ClientConfig, CogniteClient
from cognite.client.credentials import OAuthClientCredentials
from cognite.client.data_classes.data_modeling import Edge, EdgeId, NodeId, ViewId
from cognite.client.data_classes.filters import And, Equals, Filter, HasData, In, Or
from dotenv import find_dotenv, load_dotenv
from IPython.display import display

In [121]:
def create_client(
    cdf_project: str,
    cdf_cluster: str,
    client_id: str,
    client_secret: str,
    tenant_id: str,
    *,
    debug: bool = False,
) -> CogniteClient:
    scopes = [f"https://{cdf_cluster}.cognitedata.com/.default"]
    token_url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    credentials = OAuthClientCredentials(
        token_url=token_url,
        client_id=client_id,
        client_secret=client_secret,
        scopes=scopes,
    )
    config = ClientConfig(
        client_name="file-annotation-edges",
        project=cdf_project,
        base_url=f"https://{cdf_cluster}.cognitedata.com",
        credentials=credentials,
        debug=debug,
    )
    return CogniteClient(config)

In [122]:
env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)

client = create_client(
    os.environ["CDF_PROJECT"],
    os.environ["CDF_CLUSTER"],
    os.environ["IDP_CLIENT_ID"],
    os.environ["IDP_CLIENT_SECRET"],
    os.environ["IDP_TENANT_ID"],
)
print(client.config.project)

jib03


## Query

Set `FILE_EXTERNAL_ID` to one file to list every annotation edge that starts there (direct matches and pattern detections). Leave it empty to sample edges from the file space and the pattern-mode space, up to `MAX_EDGES`.

`ANNOTATION_TYPES` and `STATUSES` are optional. An empty list means all values.

In [123]:
FILE_INSTANCE_SPACE = "inst_location"
PATTERN_MODE_INSTANCE_SPACE = "sp_dat_pattern_mode_results"
PATTERN_SINK_EXTERNAL_ID = "pattern_detection_sink_node"

ANNOTATION_VIEW = ViewId(space="cdf_cdm", external_id="CogniteDiagramAnnotation", version="v1")
FILE_VIEW = ViewId(space="cdf_cdm", external_id="CogniteFile", version="v1")
ASSET_VIEW = ViewId(space="cdf_cdm", external_id="CogniteAsset", version="v1")

# One file's external id. Empty lists edges in both instance spaces, capped by MAX_EDGES.
FILE_EXTERNAL_ID = "file_PH-ME-P-0151-001.pdf"
# diagrams.AssetLink, diagrams.FileLink, or [] for both.
ANNOTATION_TYPES: list[str] = []
# Approved, Suggested, Rejected, or [] for all.
STATUSES: list[str] = []
# Edges written by this module. False includes manual and third-party annotations.
ONLY_MODULE_EDGES = True
MAX_EDGES = 5000

MODULE_SOURCE_USERS = ["fn_file_annotation", "fn_file_annotation_finalize"]

In [124]:
def _property_ref(view: ViewId, property_name: str) -> list[str]:
    return [view.space, f"{view.external_id}/{view.version}", property_name]


def annotation_filter(
    *,
    file_external_id: str,
    file_space: str,
    edge_spaces: Sequence[str],
    annotation_types: Sequence[str],
    statuses: Sequence[str],
    only_module_edges: bool,
) -> Filter:
    clauses: list[Filter] = [HasData(views=[ANNOTATION_VIEW])]
    if file_external_id:
        clauses.append(
            Equals(
                ["edge", "startNode"],
                {"space": file_space, "externalId": file_external_id},
            )
        )
    elif edge_spaces:
        clauses.append(Or(*[Equals(["edge", "space"], space) for space in edge_spaces]))
    if annotation_types:
        clauses.append(
            Or(
                *[
                    Equals(
                        ["edge", "type"],
                        {"space": ANNOTATION_VIEW.space, "externalId": annotation_type},
                    )
                    for annotation_type in annotation_types
                ]
            )
        )
    if statuses:
        clauses.append(In(_property_ref(ANNOTATION_VIEW, "status"), list(statuses)))
    if only_module_edges:
        clauses.append(In(_property_ref(ANNOTATION_VIEW, "sourceCreatedUser"), MODULE_SOURCE_USERS))
    return And(*clauses)


def list_annotation_edges(cdf: CogniteClient, edge_filter: Filter, limit: int) -> list[Edge]:
    result = cdf.data_modeling.instances.list(
        instance_type="edge",
        sources=[ANNOTATION_VIEW],
        filter=edge_filter,
        limit=limit,
    )
    return list(result)


def delete_edges_and_raw(
    cdf: CogniteClient,
    edges: Sequence[Edge],
    *,
    raw_db: str,
    raw_table_doc_doc: str,
    raw_table_doc_tag: str,
    raw_table_doc_pattern: str,
    delete_raw: bool,
    batch_size: int = 1000,
) -> dict[str, int]:
    edge_ids = [EdgeId(edge.space, edge.external_id) for edge in edges]
    doc_keys: list[str] = []
    tag_keys: list[str] = []
    pattern_keys: list[str] = []
    for edge in edges:
        annotation_type = edge.type.external_id if edge.type else ""
        if edge.space == PATTERN_MODE_INSTANCE_SPACE:
            pattern_keys.append(edge.external_id)
        elif annotation_type == "diagrams.FileLink":
            doc_keys.append(edge.external_id)
        else:
            tag_keys.append(edge.external_id)

    for start in range(0, len(edge_ids), batch_size):
        cdf.data_modeling.instances.delete(edges=edge_ids[start : start + batch_size])

    raw_deleted = {"doc": 0, "tag": 0, "pattern": 0}
    if delete_raw and raw_db:
        for keys, table, label in (
            (doc_keys, raw_table_doc_doc, "doc"),
            (tag_keys, raw_table_doc_tag, "tag"),
            (pattern_keys, raw_table_doc_pattern, "pattern"),
        ):
            for start in range(0, len(keys), batch_size):
                batch = keys[start : start + batch_size]
                cdf.raw.rows.delete(db_name=raw_db, table_name=table, key=batch)
            raw_deleted[label] = len(keys)

    return {
        "edges": len(edge_ids),
        "raw_doc": raw_deleted["doc"],
        "raw_tag": raw_deleted["tag"],
        "raw_pattern": raw_deleted["pattern"],
    }


def node_names(cdf: CogniteClient, node_ids: Sequence[NodeId]) -> dict[tuple[str, str], str]:
    names: dict[tuple[str, str], str] = {}
    unique = list({(node.space, node.external_id): node for node in node_ids}.values())
    for start in range(0, len(unique), 100):
        nodes = cdf.data_modeling.instances.retrieve_nodes(
            nodes=unique[start : start + 100],
            sources=[ASSET_VIEW, FILE_VIEW],
        )
        for node in nodes:
            name = ""
            for view in (ASSET_VIEW, FILE_VIEW):
                properties = node.properties.get(view) or {}
                candidate = properties.get("name")
                if isinstance(candidate, str) and candidate:
                    name = candidate
                    break
            names[(node.space, node.external_id)] = name
    return names


def edge_row(edge: Edge, names: dict[tuple[str, str], str]) -> dict[str, object]:
    properties = dict(edge.properties.get(ANNOTATION_VIEW) or {})
    tags = properties.get("tags")
    return {
        "edge_external_id": edge.external_id,
        "edge_space": edge.space,
        "file_space": edge.start_node.space,
        "file_external_id": edge.start_node.external_id,
        "file_name": names.get((edge.start_node.space, edge.start_node.external_id), ""),
        "annotation_type": edge.type.external_id if edge.type else None,
        "status": properties.get("status"),
        "confidence": properties.get("confidence"),
        "startNodeText": properties.get("startNodeText"),
        "startNodePageNumber": properties.get("startNodePageNumber"),
        "end_space": edge.end_node.space,
        "end_external_id": edge.end_node.external_id,
        "end_name": names.get((edge.end_node.space, edge.end_node.external_id), ""),
        "points_at_sink": (
            edge.end_node.space == PATTERN_MODE_INSTANCE_SPACE and edge.end_node.external_id == PATTERN_SINK_EXTERNAL_ID
        ),
        "tags": ", ".join(tags) if isinstance(tags, list) else tags,
        "startNodeXMin": properties.get("startNodeXMin"),
        "startNodeYMin": properties.get("startNodeYMin"),
        "startNodeXMax": properties.get("startNodeXMax"),
        "startNodeYMax": properties.get("startNodeYMax"),
        "sourceCreatedUser": properties.get("sourceCreatedUser"),
        "sourceUpdatedUser": properties.get("sourceUpdatedUser"),
        "sourceCreatedTime": properties.get("sourceCreatedTime"),
        "sourceUpdatedTime": properties.get("sourceUpdatedTime"),
        "description": properties.get("description"),
        "created_time": pd.to_datetime(edge.created_time, unit="ms", utc=True),
        "last_updated_time": pd.to_datetime(edge.last_updated_time, unit="ms", utc=True),
    }

In [125]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

edge_filter = annotation_filter(
    file_external_id=FILE_EXTERNAL_ID,
    file_space=FILE_INSTANCE_SPACE,
    edge_spaces=[FILE_INSTANCE_SPACE, PATTERN_MODE_INSTANCE_SPACE],
    annotation_types=ANNOTATION_TYPES,
    statuses=STATUSES,
    only_module_edges=ONLY_MODULE_EDGES,
)
edges = list_annotation_edges(client, edge_filter, MAX_EDGES)
print(f"Retrieved {len(edges)} annotation edge(s).")
if len(edges) == MAX_EDGES:
    print("Result is capped at MAX_EDGES. Set FILE_EXTERNAL_ID or raise the cap.")

node_ids = [NodeId(space=edge.start_node.space, external_id=edge.start_node.external_id) for edge in edges] + [
    NodeId(space=edge.end_node.space, external_id=edge.end_node.external_id) for edge in edges
]
names = node_names(client, node_ids)
annotations = pd.DataFrame([edge_row(edge, names) for edge in edges])

if annotations.empty:
    print("No annotation edges matched. Check the file external id, instance spaces, and ONLY_MODULE_EDGES.")
else:
    sort_columns = [
        column
        for column in ("file_external_id", "startNodePageNumber", "startNodeText", "annotation_type")
        if column in annotations.columns
    ]
    annotations = annotations.sort_values(sort_columns, kind="stable").reset_index(drop=True)
    summary = (
        annotations.groupby(["annotation_type", "status"], dropna=False)
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["annotation_type", "status"])
    )
    display(summary)
    sink_count = int(annotations["points_at_sink"].sum())
    print(f"{sink_count} edge(s) still point at the pattern sink ({PATTERN_SINK_EXTERNAL_ID}).")
    print("Use the table index below as INSPECT_ROW to print one edge in full.")

annotations

Retrieved 29 annotation edge(s).


,annotation_type,status,count
0,diagrams.AssetLink,Approved,24
1,diagrams.AssetLink,Suggested,1
2,diagrams.FileLink,Approved,4


0 edge(s) still point at the pattern sink (pattern_detection_sink_node).
Use the table index below as INSPECT_ROW to print one edge in full.


,edge_external_id,edge_space,file_space,file_external_id,file_name,annotation_type,status,confidence,startNodeText,startNodePageNumber,end_space,end_external_id,end_name,points_at_sink,tags,startNodeXMin,startNodeYMin,startNodeXMax,startNodeYMax,sourceCreatedUser,sourceUpdatedUser,sourceCreatedTime,sourceUpdatedTime,description,created_time,last_updated_time
0,pattern:file_PH-ME-P-0151-001.pdf:23-ESDV-92501-A:assetlink:4642bc452d,inst_location,inst_location,file_PH-ME-P-0151-001.pdf,,diagrams.AssetLink,Suggested,0.60,23-ESDV-92501-A,1,inst_location,23-ESDV-92501A,,False,"PromoteAttempted, AmbiguousMatch",0.400967,0.221905,0.419504,0.237878,fn_file_annotation,fn_file_annotation,2026-10-02T13:30:58+00:00,2026-10-02T13:30:58+00:00,AmbiguousMatch alternatives: inst_location/23-ESDV-92501A-PST,2026-10-02 13:31:29.745000+00:00,2026-10-02 13:31:29.745000+00:00
1,file_PH-ME-P-0151-001.pdf:23-FO-92445:23-FO-92445:be97c9aea8,inst_location,inst_location,file_PH-ME-P-0151-001.pdf,,diagrams.AssetLink,Approved,1.00,23-FO-92445,1,inst_location,23-FO-92445,,False,None,0.369132,0.101255,0.386258,0.116942,fn_file_annotation,fn_file_annotation,2026-10-02T13:30:58+00:00,2026-10-02T13:30:58+00:00,None,2026-10-02 13:30:58.518000+00:00,2026-10-02 13:30:58.518000+00:00
2,file_PH-ME-P-0151-001.pdf:23-FT-92512:23-FT-92512:01451c3892,inst_location,inst_location,file_PH-ME-P-0151-001.pdf,,diagrams.AssetLink,Approved,1.00,23-FT-92512,1,inst_location,23-FT-92512,,False,None,0.246021,0.205647,0.263349,0.220479,fn_file_annotation,fn_file_annotation,2026-10-02T13:30:58+00:00,2026-10-02T13:30:58+00:00,None,2026-10-02 13:30:58.518000+00:00,2026-10-02 13:30:58.518000+00:00
3,file_PH-ME-P-0151-001.pdf:23-PT-92512:23-PT-92512:9401d71f6d,inst_location,inst_location,file_PH-ME-P-0151-001.pdf,,diagrams.AssetLink,Approved,1.00,23-PT-92512,1,inst_location,23-PT-92512,,False,None,0.269998,0.205647,0.288132,0.221620,fn_file_annotation,fn_file_annotation,2026-10-02T13:30:58+00:00,2026-10-02T13:30:58+00:00,None,2026-10-02 13:30:58.518000+00:00,2026-10-02 13:30:58.518000+00:00
4,file_PH-ME-P-0151-001.pdf:23-TT-92502:23-TT-92502:86a7243ff8,inst_location,inst_location,file_PH-ME-P-0151-001.pdf,,diagrams.AssetLink,Approved,1.00,23-TT-92502,1,inst_location,23-TT-92502,,False,None,0.482168,0.212208,0.500302,0.227610,fn_file_annotation,fn_file_annotation,2026-10-02T13:30:58+00:00,2026-10-02T13:30:58+00:00,None,2026-10-02 13:30:58.518000+00:00,2026-10-02 13:30:58.518000+00:00
5,file_PH-ME-P-0151-001.pdf:23-TT-92504:23-TT-92504:abe2aa7d24,inst_location,inst_location,file_PH-ME-P-0151-001.pdf,,diagrams.AssetLink,Approved,1.00,23-TT-92504,1,inst_location,23-TT-92504,,False,None,0.660488,0.653736,0.677413,0.668853,fn_file_annotation,fn_file_annotation,2026-10-02T13:30:58+00:00,2026-10-02T13:30:58+00:00,None,2026-10-02 13:30:58.518000+00:00,2026-10-02 13:30:58.518000+00:00
6,file_PH-ME-P-0151-001.pdf:23-TT-92512:23-TT-92512:2839566573,inst_location,inst_location,file_PH-ME-P-0151-001.pdf,,diagrams.AssetLink,Approved,1.00,23-TT-92512,1,inst_location,23-TT-92512,,False,None,0.295990,0.205647,0.314125,0.221620,fn_file_annotation,fn_file_annotation,2026-10-02T13:30:58+00:00,2026-10-02T13:30:58+00:00,None,2026-10-02 13:30:58.518000+00:00,2026-10-02 13:30:58.518000+00:00
7,file_PH-ME-P-0151-001.pdf:23-YSV-92445:23-YSV-92445:eb20b767c2,inst_location,inst_location,file_PH-ME-P-0151-001.pdf,,diagrams.AssetLink,Approved,1.00,23-YSV-92445,1,inst_location,23-YSV-92445,,False,None,0.327624,0.112949,0.344348,0.128351,fn_file_annotation,fn_file_annotation,2026-10-02T13:30:58+00:00,2026-10-02T13:30:58+00:00,None,2026-10-02 13:30:58.518000+00:00,2026-10-02 13:30:58.518000+00:00
8,file_PH-ME-P-0151-001.pdf:45-FE-92506:45-FE-92506:a61470fb66,inst_location,inst_location,file_PH-ME-P-0151-001.pdf,,diagrams.AssetLink,Approved,1.00,45-FE-92506,1,inst_location,45-FE-92506,,False,None,0.380415,0.713634,0.397340,0.729321,fn_file_annotation,fn_file_annotation,2026-10-02T13:30:58+00:00,2026-10-02T13:30:58+00:00,N

## Narrow the table

`TEXT_CONTAINS` filters the rows already retrieved. It matches the text detected inside the bounding box.

In [126]:
TEXT_CONTAINS = ""

matched = annotations
if TEXT_CONTAINS and not annotations.empty:
    matched = annotations[annotations["startNodeText"].fillna("").str.contains(TEXT_CONTAINS, case=False, regex=False)]
print(f"{len(matched)} row(s)")
matched

29 row(s)


,edge_external_id,edge_space,file_space,file_external_id,file_name,annotation_type,status,confidence,startNodeText,startNodePageNumber,end_space,end_external_id,end_name,points_at_sink,tags,startNodeXMin,startNodeYMin,startNodeXMax,startNodeYMax,sourceCreatedUser,sourceUpdatedUser,sourceCreatedTime,sourceUpdatedTime,description,created_time,last_updated_time
0,pattern:file_PH-ME-P-0151-001.pdf:23-ESDV-92501-A:assetlink:4642bc452d,inst_location,inst_location,file_PH-ME-P-0151-001.pdf,,diagrams.AssetLink,Suggested,0.60,23-ESDV-92501-A,1,inst_location,23-ESDV-92501A,,False,"PromoteAttempted, AmbiguousMatch",0.400967,0.221905,0.419504,0.237878,fn_file_annotation,fn_file_annotation,2026-10-02T13:30:58+00:00,2026-10-02T13:30:58+00:00,AmbiguousMatch alternatives: inst_location/23-ESDV-92501A-PST,2026-10-02 13:31:29.745000+00:00,2026-10-02 13:31:29.745000+00:00
1,file_PH-ME-P-0151-001.pdf:23-FO-92445:23-FO-92445:be97c9aea8,inst_location,inst_location,file_PH-ME-P-0151-001.pdf,,diagrams.AssetLink,Approved,1.00,23-FO-92445,1,inst_location,23-FO-92445,,False,None,0.369132,0.101255,0.386258,0.116942,fn_file_annotation,fn_file_annotation,2026-10-02T13:30:58+00:00,2026-10-02T13:30:58+00:00,None,2026-10-02 13:30:58.518000+00:00,2026-10-02 13:30:58.518000+00:00
2,file_PH-ME-P-0151-001.pdf:23-FT-92512:23-FT-92512:01451c3892,inst_location,inst_location,file_PH-ME-P-0151-001.pdf,,diagrams.AssetLink,Approved,1.00,23-FT-92512,1,inst_location,23-FT-92512,,False,None,0.246021,0.205647,0.263349,0.220479,fn_file_annotation,fn_file_annotation,2026-10-02T13:30:58+00:00,2026-10-02T13:30:58+00:00,None,2026-10-02 13:30:58.518000+00:00,2026-10-02 13:30:58.518000+00:00
3,file_PH-ME-P-0151-001.pdf:23-PT-92512:23-PT-92512:9401d71f6d,inst_location,inst_location,file_PH-ME-P-0151-001.pdf,,diagrams.AssetLink,Approved,1.00,23-PT-92512,1,inst_location,23-PT-92512,,False,None,0.269998,0.205647,0.288132,0.221620,fn_file_annotation,fn_file_annotation,2026-10-02T13:30:58+00:00,2026-10-02T13:30:58+00:00,None,2026-10-02 13:30:58.518000+00:00,2026-10-02 13:30:58.518000+00:00
4,file_PH-ME-P-0151-001.pdf:23-TT-92502:23-TT-92502:86a7243ff8,inst_location,inst_location,file_PH-ME-P-0151-001.pdf,,diagrams.AssetLink,Approved,1.00,23-TT-92502,1,inst_location,23-TT-92502,,False,None,0.482168,0.212208,0.500302,0.227610,fn_file_annotation,fn_file_annotation,2026-10-02T13:30:58+00:00,2026-10-02T13:30:58+00:00,None,2026-10-02 13:30:58.518000+00:00,2026-10-02 13:30:58.518000+00:00
5,file_PH-ME-P-0151-001.pdf:23-TT-92504:23-TT-92504:abe2aa7d24,inst_location,inst_location,file_PH-ME-P-0151-001.pdf,,diagrams.AssetLink,Approved,1.00,23-TT-92504,1,inst_location,23-TT-92504,,False,None,0.660488,0.653736,0.677413,0.668853,fn_file_annotation,fn_file_annotation,2026-10-02T13:30:58+00:00,2026-10-02T13:30:58+00:00,None,2026-10-02 13:30:58.518000+00:00,2026-10-02 13:30:58.518000+00:00
6,file_PH-ME-P-0151-001.pdf:23-TT-92512:23-TT-92512:2839566573,inst_location,inst_location,file_PH-ME-P-0151-001.pdf,,diagrams.AssetLink,Approved,1.00,23-TT-92512,1,inst_location,23-TT-92512,,False,None,0.295990,0.205647,0.314125,0.221620,fn_file_annotation,fn_file_annotation,2026-10-02T13:30:58+00:00,2026-10-02T13:30:58+00:00,None,2026-10-02 13:30:58.518000+00:00,2026-10-02 13:30:58.518000+00:00
7,file_PH-ME-P-0151-001.pdf:23-YSV-92445:23-YSV-92445:eb20b767c2,inst_location,inst_location,file_PH-ME-P-0151-001.pdf,,diagrams.AssetLink,Approved,1.00,23-YSV-92445,1,inst_location,23-YSV-92445,,False,None,0.327624,0.112949,0.344348,0.128351,fn_file_annotation,fn_file_annotation,2026-10-02T13:30:58+00:00,2026-10-02T13:30:58+00:00,None,2026-10-02 13:30:58.518000+00:00,2026-10-02 13:30:58.518000+00:00
8,file_PH-ME-P-0151-001.pdf:45-FE-92506:45-FE-92506:a61470fb66,inst_location,inst_location,file_PH-ME-P-0151-001.pdf,,diagrams.AssetLink,Approved,1.00,45-FE-92506,1,inst_location,45-FE-92506,,False,None,0.380415,0.713634,0.397340,0.729321,fn_file_annotation,fn_file_annotation,2026-10-02T13:30:58+00:00,2026-10-02T13:30:58+00:00,N

## One edge

Set `INSPECT_ROW` to a table index from the results above (for example `0`). The cell re-retrieves that edge and prints every property from every source view, so nothing is truncated.

Note: `isAssetVerified` belongs to diagram-parsing entities, not to `CogniteDiagramAnnotation` edges written by this module. The cell still searches for it among the edge properties.

In [127]:
INSPECT_ROW = 0

if annotations.empty:
    print("No annotation edges to inspect.")
elif INSPECT_ROW not in annotations.index:
    print(f"INSPECT_ROW must be one of: {list(annotations.index)}")
else:
    row = annotations.loc[INSPECT_ROW]
    edge_id = EdgeId(space=str(row["edge_space"]), external_id=str(row["edge_external_id"]))
    print(f"row:              {INSPECT_ROW}")
    print(f"edge_space:       {edge_id.space}")
    print(f"edge_external_id: {edge_id.external_id}")

    retrieved = client.data_modeling.instances.retrieve(
        edges=[edge_id],
        sources=[ANNOTATION_VIEW],
    )
    selected = next(iter(retrieved.edges), None) if retrieved.edges else None
    if selected is None:
        # Fall back to the listed edge if retrieve returns nothing.
        selected = next(
            (
                edge
                for edge in edges
                if edge.external_id == edge_id.external_id and edge.space == edge_id.space
            ),
            None,
        )

    if selected is None:
        print("That edge is not in the current result.")
    else:
        print(f"type:  {selected.type.external_id if selected.type else None}")
        print(f"start: ({selected.start_node.space}, {selected.start_node.external_id})")
        print(f"end:   ({selected.end_node.space}, {selected.end_node.external_id})")

        print("\n=== Full edge dump ===")
        pprint(selected.dump(camel_case=True), sort_dicts=False)

        print("\n=== Properties by view ===")
        all_properties: dict[str, object] = {}
        for view_id, view_properties in (selected.properties or {}).items():
            view_label = (
                f"{view_id.space}/{view_id.external_id}/{view_id.version}"
                if hasattr(view_id, "space")
                else str(view_id)
            )
            props = dict(view_properties or {})
            print(f"\n[{view_label}] ({len(props)} properties)")
            for key in sorted(props):
                print(f"  {key}: {props[key]!r}")
                all_properties[key] = props[key]

        print("\n=== Flat property lookup ===")
        for key in sorted(all_properties):
            print(f"{key}: {all_properties[key]!r}")

        print("\n=== isAssetVerified ===")
        if "isAssetVerified" in all_properties:
            print(f"isAssetVerified: {all_properties['isAssetVerified']!r}")
        else:
            print(
                "Not present on this CogniteDiagramAnnotation edge. "
                "That field is used by diagram-parsing entities, not by file-annotation edges."
            )

        print("\n=== JSON ===")
        print(json.dumps(selected.dump(camel_case=True), indent=2, default=str))

row:              0
edge_space:       inst_location
edge_external_id: pattern:file_PH-ME-P-0151-001.pdf:23-ESDV-92501-A:assetlink:4642bc452d
type:  diagrams.AssetLink
start: (inst_location, file_PH-ME-P-0151-001.pdf)
end:   (inst_location, 23-ESDV-92501A)

=== Full edge dump ===
{'space': 'inst_location',
 'externalId': 'pattern:file_PH-ME-P-0151-001.pdf:23-ESDV-92501-A:assetlink:4642bc452d',
 'version': 1,
 'lastUpdatedTime': 1790947889745,
 'createdTime': 1790947889745,
 'instanceType': 'edge',
 'properties': {'cdf_cdm': {'CogniteDiagramAnnotation/v1': {'status': 'Suggested',
                                                            'confidence': 0.6,
                                                            'name': 'file_PH-ME-P-0151-001.pdf',
                                                            'tags': ['PromoteAttempted',
                                                                     'AmbiguousMatch'],
                                                            'd

## Clear annotations

Deletes `CogniteDiagramAnnotation` edges created by this module (`fn_file_annotation`), matching the finalize clean behavior.

Pick one mode:

- `CLEAR_MODE = "file"` — delete every module edge that starts at `CLEAR_FILE_EXTERNAL_ID` (file instance space and pattern-mode space).
- `CLEAR_MODE = "space"` — delete every module edge stored in `CLEAR_EDGE_SPACE` (for example `inst_location` or `sp_dat_pattern_mode_results`).

Set `CONFIRM_DELETE = True` to run the delete. With it left `False`, the cell only lists what would be removed. `DELETE_RAW_ROWS` also removes the matching RAW audit rows.

In [128]:
# One of: "file" or "space"
CLEAR_MODE = "file"

# Used when CLEAR_MODE == "file"
CLEAR_FILE_EXTERNAL_ID = FILE_EXTERNAL_ID  # e.g. "file_PH-ME-P-0151-001.pdf"
CLEAR_FILE_SPACE = FILE_INSTANCE_SPACE

# Used when CLEAR_MODE == "space"
# Typical values: FILE_INSTANCE_SPACE or PATTERN_MODE_INSTANCE_SPACE
CLEAR_EDGE_SPACE = FILE_INSTANCE_SPACE

# Only edges written by cdf_file_annotation. Set False to include manual/third-party too.
CLEAR_ONLY_MODULE_EDGES = True
CLEAR_MAX_EDGES = 50000

# Matching ApplyService clean: also remove RAW rows keyed by the edge external ids.
DELETE_RAW_ROWS = True
RAW_DB = "db_file_annotation"
RAW_TABLE_DOC_DOC = "annotation_documents_docs"
RAW_TABLE_DOC_TAG = "annotation_documents_tags"
RAW_TABLE_DOC_PATTERN = "annotation_documents_patterns"

# Safety latch — leave False to preview, set True to delete.
CONFIRM_DELETE = True

In [129]:
if CLEAR_MODE == "file":
    if not CLEAR_FILE_EXTERNAL_ID:
        raise ValueError('Set CLEAR_FILE_EXTERNAL_ID when CLEAR_MODE is "file".')
    clear_filter = annotation_filter(
        file_external_id=CLEAR_FILE_EXTERNAL_ID,
        file_space=CLEAR_FILE_SPACE,
        edge_spaces=[CLEAR_FILE_SPACE, PATTERN_MODE_INSTANCE_SPACE],
        annotation_types=[],
        statuses=[],
        only_module_edges=CLEAR_ONLY_MODULE_EDGES,
    )
    scope_label = f"file ({CLEAR_FILE_SPACE}, {CLEAR_FILE_EXTERNAL_ID})"
elif CLEAR_MODE == "space":
    if not CLEAR_EDGE_SPACE:
        raise ValueError('Set CLEAR_EDGE_SPACE when CLEAR_MODE is "space".')
    clear_filter = annotation_filter(
        file_external_id="",
        file_space=CLEAR_FILE_SPACE,
        edge_spaces=[CLEAR_EDGE_SPACE],
        annotation_types=[],
        statuses=[],
        only_module_edges=CLEAR_ONLY_MODULE_EDGES,
    )
    scope_label = f"edge space {CLEAR_EDGE_SPACE}"
else:
    raise ValueError('CLEAR_MODE must be "file" or "space".')

edges_to_clear = list_annotation_edges(client, clear_filter, CLEAR_MAX_EDGES)
preview = pd.DataFrame(
    [
        {
            "edge_space": edge.space,
            "edge_external_id": edge.external_id,
            "file_external_id": edge.start_node.external_id,
            "annotation_type": edge.type.external_id if edge.type else None,
            "end_external_id": edge.end_node.external_id,
            "status": (edge.properties.get(ANNOTATION_VIEW) or {}).get("status"),
            "startNodeText": (edge.properties.get(ANNOTATION_VIEW) or {}).get("startNodeText"),
        }
        for edge in edges_to_clear
    ]
)
print(f"Found {len(edges_to_clear)} annotation edge(s) for {scope_label}.")
if len(edges_to_clear) == CLEAR_MAX_EDGES:
    print("Result is capped at CLEAR_MAX_EDGES. Raise the cap or clear in smaller scopes.")
display(preview.head(50))

if not edges_to_clear:
    print("Nothing to delete.")
elif not CONFIRM_DELETE:
    print("Preview only. Set CONFIRM_DELETE = True and re-run this cell to delete.")
else:
    deleted = delete_edges_and_raw(
        client,
        edges_to_clear,
        raw_db=RAW_DB,
        raw_table_doc_doc=RAW_TABLE_DOC_DOC,
        raw_table_doc_tag=RAW_TABLE_DOC_TAG,
        raw_table_doc_pattern=RAW_TABLE_DOC_PATTERN,
        delete_raw=DELETE_RAW_ROWS,
    )
    print(
        f"Deleted {deleted['edges']} edge(s). "
        f"RAW rows removed: doc={deleted['raw_doc']}, tag={deleted['raw_tag']}, pattern={deleted['raw_pattern']}."
    )

Found 29 annotation edge(s) for file (inst_location, file_PH-ME-P-0151-001.pdf).


,edge_space,edge_external_id,file_external_id,annotation_type,end_external_id,status,startNodeText
0,inst_location,file_PH-ME-P-0151-001.pdf:23-FO-92445:23-FO-92445:be97c9aea8,file_PH-ME-P-0151-001.pdf,diagrams.AssetLink,23-FO-92445,Approved,23-FO-92445
1,inst_location,file_PH-ME-P-0151-001.pdf:23-FT-92512:23-FT-92512:01451c3892,file_PH-ME-P-0151-001.pdf,diagrams.AssetLink,23-FT-92512,Approved,23-FT-92512
2,inst_location,file_PH-ME-P-0151-001.pdf:23-HV-92445:HV-92445:52cf0609a2,file_PH-ME-P-0151-001.pdf,diagrams.AssetLink,23-HV-92445,Approved,HV-92445
3,inst_location,file_PH-ME-P-0151-001.pdf:23-PDI-92501:PDI-92501:c289645123,file_PH-ME-P-0151-001.pdf,diagrams.AssetLink,23-PDI-92501,Approved,PDI-92501
4,inst_location,file_PH-ME-P-0151-001.pdf:23-PT-92512:23-PT-92512:9401d71f6d,file_PH-ME-P-0151-001.pdf,diagrams.AssetLink,23-PT-92512,Approved,23-PT-92512
5,inst_location,file_PH-ME-P-0151-001.pdf:23-TIC-92504:TIC-92504:322e1a9712,file_PH-ME-P-0151-001.pdf,diagrams.AssetLink,23-TIC-92504,Approved,TIC-92504
6,inst_location,file_PH-ME-P-0151-001.pdf:23-TT-92502:23-TT-92502:86a7243ff8,file_PH-ME-P-0151-001.pdf,diagrams.AssetLink,23-TT-92502,Approved,23-TT-92502
7,inst_location,file_PH-ME-P-0151-001.pdf:23-TT-92504:23-TT-92504:abe2aa7d24,file_PH-ME-P-0151-001.pdf,diagrams.AssetLink,23-TT-92504,Approved,23-TT-92504
8,inst_location,file_PH-ME-P-0151-001.pdf:23-TT-92512:23-TT-92512:2839566573,file_PH-ME-P-0151-001.pdf,diagrams.AssetLink,23-TT-92512,Approved,23-TT-92512
9,inst_location,file_PH-ME-P-0151-001.pdf:23-TT-92512:TT-92512:6dc66d0df0,file_PH-ME-P-0151-001.pdf,diagrams.AssetLink,23-TT-92512,Approved,TT-92512


Deleted 29 edge(s). RAW rows removed: doc=4, tag=25, pattern=0.
